In [124]:
import pandas as pd
import os
import json
import datetime
from sklearn.model_selection import train_test_split
os.chdir("/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis")

If we want to train a classifier for level $l \in [1,2,3,4]$, we can either use descriptions of level $l$, or those of a lower level. 

We did the first thing earlier, lets do the second thing. 

I.e. genereate data for a classifier desc -> [A, B, C, ...] and use the descriptions of the lowest level of the classes [A, B, C, ...]. 

In [101]:
%pwd

'/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis'

In [102]:
nace_description_path = "data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv"
nace_descriptions = pd.read_csv(nace_description_path, sep="\t")

In [103]:
def get_sublevels(nace_class, level): 
    nace_class_temp = nace_class
    nace_id = nace_descriptions[nace_descriptions["CODE"] == nace_class_temp]["ID"].iloc[0]
    nace_class_lvl_2 = []
    nace_class_lvl_3 = []
    nace_class_lvl_4 = []

    for _, row in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id].iterrows(): 
        nace_id_temp = row["ID"]
        nace_class_lvl_2.append(row["CODE"])
        for _, row_2 in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id_temp].iterrows(): 
            nace_id_temp_temp = row_2["ID"]
            nace_class_lvl_3.append(row_2["CODE"])
            for _, row_3 in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id_temp_temp].iterrows(): 
                nace_class_lvl_4.append(row_3["CODE"])
        
    if level == 2: 
        return nace_class_lvl_2
    if level == 3: 
        return nace_class_lvl_3
    if level == 4: 
        return nace_class_lvl_4

In [104]:
synthetic_data_path = "data/synthetic_data/"

## Classification of level 1

In [105]:
level_classification = 1
level_data = 3
generated_classes = ["A", "B", "C", "F", "J"]
head_nace_code = None

In [142]:
date = "20251221"
date_2 = "20251222"

In [ ]:
data = {}

for generate_nace_class in generated_classes:

    subclasses_2 = get_sublevels(generate_nace_class, level=2) 
    description_classes = []
    descriptions = []
    prompts = {}

    # read sublevels
    for subclass_2 in subclasses_2:

        subclasses_3 = get_sublevels(subclass_2, level=2) 
        for subclass_3 in subclasses_3: 

            class_path = os.path.join(synthetic_data_path, f"data_{date}__level_{level_data}__subclasses_{subclass_2}", f"class_{subclass_3}.csv")    
            if not os.path.exists(class_path): 
                class_path = os.path.join(synthetic_data_path, f"data_{date_2}__level_{level_data}__subclasses_{subclass_2}", f"class_{subclass_3}.csv")    
                
            if os.path.exists(class_path): 
                df = pd.read_csv(class_path)
                description_classes.append(subclass_3)
                
                df["label"] = generate_nace_class
                df["label_lvl_3"] = subclass_3
                df = df.rename(columns={subclass_3: "text"})
                descriptions.append(df)

            config_path = os.path.join(synthetic_data_path, f"data_{date}__level_{level_data}__subclasses_{subclass_2}", f"config.json")   
            if not os.path.exists(class_path): 
                config_path = os.path.join(synthetic_data_path, f"data_{date_2}__level_{level_data}__subclasses_{subclass_2}", f"config.json")   

            if os.path.exists(config_path): 
                with open(config_path, "r") as f: 
                    config = json.load(f)
                    prompts.update(config["prompts"])

    print(generate_nace_class, subclasses_3, description_classes)
    print(len(descriptions))
    if len(descriptions) > 0: 
        data[generate_nace_class] = {
            "description_level": level_data, 
            "description_classes": description_classes, 
            "data": pd.concat(descriptions),
            "prompts": prompts, 
            "system_prompt": config["system_prompt"]
        }

        # results = {
        #     "data": data,
        #     "prompt": res[0],
        #     "system_prompt": res[0].messages[0].content,
        #     "user_prompt": res[0].messages[1].content,
        #     "output": examples
        # }

A ['03.1', '03.2'] ['01.1', '01.2', '01.3', '01.4', '02.1', '02.2', '02.3', '03.1', '03.2']
9
B ['09.1', '09.9'] ['05.1', '05.2', '06.1', '06.2', '07.1', '07.2']
6
C ['33.1', '33.2'] ['20.1', '20.2', '20.3', '20.4', '20.5', '20.6', '21.1', '21.2']
8
F ['43.1', '43.2', '43.3', '43.9'] ['41.1', '41.2', '42.1', '42.2', '42.9', '43.1', '43.2', '43.3', '43.9']
9
J ['63.1', '63.9'] ['58.1', '58.2', '63.1', '63.9']
4


In [118]:
df_full = []
for k in data: 

    df_full.append(data[k]["data"])

df_full = pd.concat(df_full)
df_full

,text,label,label_lvl_3
0,"In the past year, our commitment to sustainabl...",A,01.1
1,Our operations are deeply rooted in the rural ...,A,01.1
2,Weather patterns have played a pivotal role in...,A,01.1
3,"Planning cycles are inherently seasonal, which...",A,01.1
4,Input cost volatility remains a significant ch...,A,01.1
...,...,...,...
895,Our commitment to continuous improvement is re...,J,63.9
896,The diversity of our operations across rural r...,J,63.9
897,"As we look ahead, maintaining strong relations...",J,63.9
898,Our workforce remains our greatest asset. This...,J,63.9


### test train val split and store... 

In [126]:
date = datetime.datetime.now().strftime("%Y%m%d")

store_path = os.path.join(synthetic_data_path, f"data_{date}__level_{level_classification}__subclasses_{head_nace_code}__level_descriptions_{level_data}")    

os.makedirs(store_path)
store_path

'data/synthetic_data/data_20251222__level_1__subclasses_None__level_descriptions_3'

In [ ]:
for k in data: 
    data[k]["data"].to_csv(os.path.join(store_path, f"class_{k}.csv"))

                                                  text label label_lvl_3
0    In the past year, our commitment to sustainabl...     A        01.1
1    Our operations are deeply rooted in the rural ...     A        01.1
2    Weather patterns have played a pivotal role in...     A        01.1
3    Planning cycles are inherently seasonal, which...     A        01.1
4    Input cost volatility remains a significant ch...     A        01.1
..                                                 ...   ...         ...
894  The volatility of input costs remains a challe...     A        03.2
895  Our workforce is our greatest asset, and we ha...     A        03.2
896  Innovation is at the forefront of our operatio...     A        03.2
897  Looking ahead, we remain optimistic about our ...     A        03.2
898  In summary, this year has been marked by both ...     A        03.2

[8101 rows x 3 columns]
                                                  text label label_lvl_3
0    Our strategic focus o

In [127]:
train_df, temp_df = train_test_split(df_full, test_size=0.4, random_state=42, stratify=df_full["label"])
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

df_full.to_csv(os.path.join(store_path, "synthetic_data_full.csv"), index=False)
train_df.to_csv(os.path.join(store_path, "train_data.csv"), index=False)
test_df.to_csv(os.path.join(store_path, "test_data.csv"), index=False)
val_df.to_csv(os.path.join(store_path, "val_data.csv"), index=False)

In [ ]:
config = data.copy()
for k in config:
    config[k]["data"] = None

config_path = os.path.join(store_path, "config.json")

with open(config_path, "w") as f: 
    json.dump(config, f)

## Classification of level 2

A -> 1,2,3
B -> 5,6,7

In [176]:
date = "20251221"
date_2 = "20251222"

In [190]:
divisions = {
    "A": ["1", "2", "3"],  
    "B": ['5', '6', '7'],
    "C": ['20', '21'], 
    "F": ["41", "42", "43"], 
    "J": ["58", "63"],
    }
classes = {
    "1": ["01.1","01.2","01.3","01.4"],
    "2": ['02.1', '02.2', '02.3'],
    "3": ['03.1', '03.2'],
    "5": ['05.1', '05.2'],
    "6": ['06.1', '06.2'],
    "7": ['07.1', '07.2'],
    "20": ['20.1', '20.2', '20.3', '20.4', '20.5', '20.6'],
    "21": ['21.1', '21.2'],
    "41": ["41.1", "41.2"], 
    "42": ["42.1", "42.2","42.9"], 
    "43": ["43.1", "43.2", "43.3","43.9"], 
    "58": ['58.1', '58.2'],
    "63": ['63.1', '63.9'],
    }

In [191]:
level_classification = 2
level_data = 3
data = {}

for head_nace_code in divisions:
    # read sublevels

    print(head_nace_code)
    
    subclasses_2 = divisions[head_nace_code]
    
    description_classes = []
    descriptions = []
    prompts = {}

    # read sublevels
    for subclass_2 in subclasses_2:

        subclasses_3 = classes[subclass_2]
        for subclass_3 in subclasses_3: 

            class_path = os.path.join(synthetic_data_path, f"data_{date}__level_{level_data}__subclasses_{subclass_2}", f"class_{subclass_3}.csv")    
            if not os.path.exists(class_path): 
                class_path = os.path.join(synthetic_data_path, f"data_{date_2}__level_{level_data}__subclasses_{subclass_2}", f"class_{subclass_3}.csv")    
                
            if os.path.exists(class_path): 
                df = pd.read_csv(class_path)
                description_classes.append(subclass_3)
                
                df["label"] = subclass_2
                df["label_lvl_3"] = subclass_3
                df = df.rename(columns={subclass_3: "text"})
                descriptions.append(df)

            config_path = os.path.join(synthetic_data_path, f"data_{date}__level_{level_data}__subclasses_{subclass_2}", f"config.json")   
            if not os.path.exists(class_path): 
                config_path = os.path.join(synthetic_data_path, f"data_{date_2}__level_{level_data}__subclasses_{subclass_2}", f"config.json")   

            if os.path.exists(config_path): 
                with open(config_path, "r") as f: 
                    config = json.load(f)
                    prompts.update(config["prompts"])

    df_full = pd.concat(descriptions)

    date_store = datetime.datetime.now().strftime("%Y%m%d")
    store_path = os.path.join(synthetic_data_path, f"data_{date_store}__level_{level_classification}__subclasses_{head_nace_code}__level_descriptions_{level_data}")    
    os.makedirs(store_path, exist_ok=True)

    for k in set(df_full["label"]): 
        df_full[df_full["label"] == k].to_csv(os.path.join(store_path, f"class_{k}.csv"))

    train_df, temp_df = train_test_split(df_full, test_size=0.4, random_state=42, stratify=df_full["label"])
    test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

    df_full.to_csv(os.path.join(store_path, "synthetic_data_full.csv"), index=False)
    train_df.to_csv(os.path.join(store_path, "train_data.csv"), index=False)
    test_df.to_csv(os.path.join(store_path, "test_data.csv"), index=False)
    val_df.to_csv(os.path.join(store_path, "val_data.csv"), index=False)
    
    config = {"description_classes": description_classes,"prompts": prompts}

    config_path = os.path.join(store_path, "config.json")

    with open(config_path, "w") as f: 
        json.dump(config, f)    

A
B
C
F
J
